# Figure: tree-accelerated SVGD vs. exact

Loads `results/svgd/convergence.json` (`bench/svgd/convergence_vs_exact.py`). Never recomputes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

_here = Path.cwd()
_candidates = [_here / "results" / "svgd", _here.parents[1] / "results" / "svgd"]
RESULTS = next((c for c in _candidates if c.exists()), _candidates[0])
FIGDIR = RESULTS.parents[1] / "examples" / "differentiable_paper" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
BASELINE = "#8C8C8C"


In [ ]:
data = json.loads((RESULTS / "convergence.json").read_text())
meta = data["metadata"]; records = data["records"]
targets = sorted({r["target"] for r in records})
thetas = sorted({r["theta"] for r in records})
print(f"device={meta['device_kind']} jax={meta['jax_version']} commit={meta.get('git_commit')}")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
for i, tname in enumerate(targets):
    xs, ys = [], []
    for th in thetas:
        rec = next((r for r in records if r["target"] == tname and r["theta"] == th), None)
        if rec is None:
            continue
        xs.append(th); ys.append(max(rec["mmd_to_exact"], 1e-12))
    ax.plot(xs, ys, "o-", color=PALETTE[i % len(PALETTE)], label=tname)
ax.set_yscale("log")
ax.set_xlabel(r"opening angle $\theta$")
ax.set_ylabel("MMD(tree SVGD, exact SVGD)")
ax.set_title("Tree-accelerated SVGD vs. exact, across theta")
ax.legend()
out = FIGDIR / "fig_svgd_convergence.pdf"
fig.savefig(out); fig.savefig(out.with_suffix(".png"))
print("wrote", out)
